In [ ]:
# SENTIMENT ANALYSIS FOR CUSTOMER REVIEWS WITH CNN-LSTM

# We use both because they have complementary strengths for text classification.
# CNNs are good at detecting local patterns and phrases.
# LSTMs are good at understanding sequence and context.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load dataset, keep only columns that we need, clean data, and create binary sentiment target variabel, then seperate text and target
# Then we will split the data into training and testing
# Then use tokenizer to convert words(NNs dont understand words) into integers then convert all text to sequences
# Then padding since RNNs require all sequences to be the same length.

vocab_size = 5000 # Maz words to keep
embedding_dim = 64 # size of each word vector., love → [0.2,0.8,0.4,...], The model learns relationships form this
max_length = 200 # Maximum sentence length.
padding_type = 'post' # Controls whee the padding/ 0s go, in this case after
trunc_type = 'post' # Controls where words are removed if the sentence is too long
oov_tok = '<OOV>' # # Token for words not in the top-5000 vocabulary
training_portion = .8



df = pd.read_csv("GrammarandProductReviews.xlsx - GrammarandProductReviews.csv")

# df = df.dropna()
# df.isnull().sum()

# Create target variable for sentiment
# Lamba a small anonymous function that you write in one line instead of using def, format is lambda parameters: expression
# We also use a ternary conditional expression which is a one-line version of if-else
# Instead of :
# if rating >= 3:
#     return 1
# else:
#     return 0

# We have 1 if rating >= 3 else 0, which fromat is value_if_true if condition else value_if_false


df['sentiment'] = df['reviews.rating'].apply(lambda rating: 1 if rating >=3 else 0)
print(df['sentiment'])

# Seperating text and sentiment(target)
X = df['reviews.text'].astype(str)
y = df['sentiment']

# Split into training and test
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

# Tokenize text
vocab_size = 10000 # Means only keep 10,000 of the most common words
oov_tok ="<OOV>" # The words not in the 10,000 get mapped to this special token

tokenzier = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenzier.fit_on_texts(X_train) # Where the tokenizer learns the vocab

# Convert reviews into sequences
X_train_seq = tokenzier.texts_to_sequences(X_train) # Takes every review in X_train_seq and repalces each word with its intenger ID
X_test_seq = tokenzier.texts_to_sequences(X_test)

# Padding the sequences so they are the same length
max_length = 100 # Every review should be 100 tokens long
padding_type ='post' # Padding(adding of zero) happens at the end of the sequence
trunc_type = "post" # If we have too many tokens, then tokens are removed from the end of the sequence

X_train_padded = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding=padding_type,
    truncating=trunc_type
                            )

X_test_padded = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding=padding_type,
    truncating=trunc_type
                            )

print(X_train_padded[10])


0        1
1        1
2        1
3        0
4        0
        ..
9996     1
9997     1
9998     1
9999     1
10000    1
Name: sentiment, Length: 10001, dtype: int64
[   2   76   26  361 5022  148 1399   62  513 3727  134    2 1055   97
   62   58  178  194    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Bidirectional

# Here we build our CNN-LSTM Model
model = Sequential([

    # We embed each number and each numbers get 64 numbers
    Embedding(input_dim=vocab_size, output_dim=64),

    # Kernel_size= 5 so it looks a 5 words at a time, 64 filters and each learns a different pattern(find important phrases)
    # Filters are phrase detectos so if it sees somehting like rlly good, very good, it produces a high score
    # ANd there are 64 different phrase/pattern detectors that give a score and in the next step we keep the strongest ones
    Conv1D(filters=64, kernel_size=5, activation='relu'),

    #Looks at chunks of 2 and keeps the max from each chunk
    MaxPooling1D(2),

    Bidirectional(LSTM(64)),

    Dropout(0.3),

    Dense(1, activation='sigmoid')

])

# COmpile model
model.compile(
    optimizer='adam',
    loss = 'binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Add early stopping to prevent overfitting
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Fitting and testing the model
hisotry = model.fit(X_train_padded,y_train, validation_split=0.2, epochs=10, batch_size=32, callbacks=[early_stopping], verbose=1)

test_loss, test_accuracy = model.evaluate(X_test_padded, y_test)

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")


Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9619 - loss: 0.1579 - val_accuracy: 0.9600 - val_loss: 0.1311
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.9688 - loss: 0.0979 - val_accuracy: 0.9638 - val_loss: 0.1296
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.9845 - loss: 0.0512 - val_accuracy: 0.9606 - val_loss: 0.2005
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.9916 - loss: 0.0310 - val_accuracy: 0.9613 - val_loss: 0.1959
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.9964 - loss: 0.0178 - val_accuracy: 0.9600 - val_loss: 0.2101
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9580 - loss: 0.1342
Test Loss: 0.1341860294342041
Test Accuracy: 0.9580209851264954
